In [1]:
import matplotlib.pyplot as plt
plt.rcParams["font.size"] = 16

# 1.A : Model

Canva

# 1.B : Jumps (Ryu (data + fitted) + 3 trajectories (Ryu + mu2 + theta2))

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import gamma
plt.rcParams["font.size"] = 16


def proba_gamma(mu: float, theta: float, L: float) -> float:
    alpha_gamma = mu**2 / theta**2
    beta_gamma = theta**2 / mu
    return gamma.pdf(L, a=alpha_gamma, scale=beta_gamma)


# Données expérimentales
x = np.arange(0, 150, 10)
y = np.array([10, 140, 240, 200, 130, 125, 90, 60, 30, 25, 20, 10, 15, 7, 5], dtype=float)
y /= np.sum(y)  # normalisation à 1

# PDF théorique
x_fine = np.arange(0, 150, 1)

# Fit
popt, pcov = curve_fit(
    lambda x, mu, theta: proba_gamma(mu, theta, x) * 10,
    x,
    y,
    p0=[30, 15],
    bounds=(0, np.inf),
    maxfev=10000
)

mu, theta = popt

# PDF continue
p = proba_gamma(mu, theta, x_fine)

# Normalisation correcte
p /= np.sum(p)

# Passage en probabilité par bin
bin_width = 10
p_plot = p * bin_width

# Plot
plt.figure(figsize=(8,6), dpi=1200)

plt.bar(x, y, width=8, edgecolor="black",
        color="red", alpha=0.5, label="Data from Ryu et al. 2022\nExtracted from Figure 3.B")

plt.plot(x_fine, p_plot, lw=2, color="red",
         label=f"Gamma fit : (μ,θ) = ({mu:.0f},{theta:.0f})nm")

plt.xlabel("Step size (nm)")
plt.ylabel("Probability")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# 1.C : Properties of the Gamma distrib

In [3]:
import matplotlib.pyplot as plt
import numpy as np
from nucleo.simulation.probabilities import proba_gamma

plt.rcParams["font.size"] = 16

x_fine = np.arange(0, 500, 1)
params = [
    (100, 20),
    (200, 20),
    (200, 200),
]

cmap = plt.cm.Reds
colors = [cmap(0.4), cmap(0.65), cmap(0.9)]

plt.figure(figsize=(8,6), dpi=1200)

for (mu, theta), color in zip(params, colors):
    p = proba_gamma(mu=mu, theta=theta, L=x_fine)
    
    plt.plot(
        x_fine,
        p,
        lw=2.5,
        color=color,
        label=f"(μ,θ) = ({mu},{theta})"
    )

plt.xlabel(r"Step size $(\sigma)$")
plt.ylabel("Probability density")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", ncols=1)
plt.tight_layout()
plt.show()

# 1.D : Chromatin lanscapes (homogeneous + periodic + random)

In [4]:
from nucleo.simulation.chromatin import alpha_random, alpha_periodic, alpha_homogeneous
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams["font.size"] = 16

# Values in nm
s = 150
l = 10
alphao = 0
alphaf = 1
Lmin = 0
Lmax = 500
bps = 1

# Landscapes
obs_1 = alpha_homogeneous(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
obs_2 = alpha_periodic(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
obs_3 = alpha_random(s=s, l=l, alphao=alphao, alphaf=alphaf, Lmin=0, Lmax=Lmax, bps=bps)
x = np.arange(0, len(obs_1), 1)

# -------------------------------
# Homogeneous
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.plot(x, obs_1, c="b", lw=3)
plt.title("Homogeneous")
plt.xlabel("x")
plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# -------------------------------
# Periodic
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.step(x, obs_2, c="b", lw=3)
plt.title("Periodic")
plt.xlabel("x")
# plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# -------------------------------
# Random
# -------------------------------
plt.figure(figsize=(8,5), dpi=1200)

plt.step(x, obs_3, c="b", lw=3)
plt.title("Random")
plt.xlabel("x")
# plt.ylabel(r"Accessibility $\alpha$")
plt.ylim([-0.1, 1.1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# 1.E : Linker and RoadBlocks distribution

In [47]:
# Librairies
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
from polars import selectors as cs
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from nucleo.metrics.utils import listoflist_into_matrix
plt.rcParams["font.size"] = 16

# Data
root = Path("/home/nicolas/Documents/Workspace/nucleo/outputs/2026-03-05__PC/nucleo__fig1_0")
paths = [str(p) for p in root.rglob("*.parquet")]
df_sorted = (
    pl.scan_parquet(paths)
    .select(
        cs.string() | cs.boolean() |  cs.integer() |  cs.float() | 
        pl.col("s_points") | pl.col("s_distrib") | pl.col("l_points") | pl.col("l_distrib")
      )
    .collect()
    .sort(by=["landscape", "bpmin", "l"],
          descending=[False, False, False]
        )
      .filter(
          pl.col("landscape") == "random"
      )
)

# Obstacles
obs_points_data = df_sorted["s_points"].to_list()
obs_points_data = listoflist_into_matrix(obs_points_data)
obs_points = np.nanmean(obs_points_data,axis=0)

obs_distrib_data = df_sorted["s_distrib"].to_list()
obs_distrib_data = listoflist_into_matrix(obs_distrib_data)
obs_distrib = np.nanmean(obs_distrib_data,axis=0)

# To roadblocks
mask_s = (obs_distrib != 0)
obs_points = obs_points[mask_s]
obs_distrib = obs_distrib[mask_s]
roadblocks_points = obs_points//s
roadblocks_distrib = obs_distrib

# Linkers
link_points_data = df_sorted["l_points"].to_list()
link_points_data = listoflist_into_matrix(link_points_data)
link_points = np.nanmean(link_points_data,axis=0)

link_distrib_data = df_sorted["l_distrib"].to_list()
link_distrib_data = listoflist_into_matrix(link_distrib_data)
link_distrib = np.nanmean(link_distrib_data,axis=0)


# Theory
from scipy.special import factorial

def th_linkers(l, lmoy):
    A = lmoy * (1 - np.exp(-1 / lmoy))
    P = A * (np.exp(1 / lmoy) - 1) * np.exp(-l / lmoy)
    return P / np.sum(P)


def th_roadblocks(m, lmoy):
    m = np.atleast_1d(m).astype(int)
    A = np.exp(1 / lmoy) - 1
    P = np.array([1 / (lmoy**mi * A * factorial(mi)) for mi in m], dtype=float)
    return P / np.sum(P)


# Plot
fig, axes = plt.subplots(ncols=1 , nrows=2, figsize=(8,6), dpi=1200)

s = 150
lmoy = 10

theory_distrib = th_linkers(link_points, lmoy)

axes[0].plot(link_points, link_distrib, label='Simulations',
             color='lightblue', alpha=1, marker='o', lw=2)
axes[0].plot(link_points, theory_distrib, label="Theory",
             color='black', ls="--", lw=2)

axes[0].set_xlabel('Size of linkers')
axes[0].set_ylabel('Distribution')
axes[0].set_xlim([-1, 45])
# axes[0].set_ylim([-0.10, 0.25])
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

print(link_distrib, link_points)



theory_distrib = th_roadblocks(roadblocks_points, lmoy)

axes[1].plot(roadblocks_points, roadblocks_distrib, label='Simulations',
      color='darkslategray', alpha=1, marker='o', lw=2)
axes[1].plot(roadblocks_points, theory_distrib, label="Theory",
             color='black', ls="--", lw=2)

axes[1].set_xticks(np.arange(1, 6, 1, dtype=int))
axes[1].set_ylim([-0.10, 1.10])
axes[1].set_xlabel("Consecutive roadblocks")
axes[1].set_ylabel('Distribution')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

[4.7904897e-02 8.9413159e-02 8.2182057e-02 7.1407393e-02 6.4911216e-02
 6.4465061e-02 5.1834095e-02 4.9881529e-02 4.3487970e-02 3.9331961e-02
 3.4501839e-02 3.3992797e-02 3.1594794e-02 2.6253905e-02 2.5293894e-02
 2.2287318e-02 2.0977415e-02 1.8578691e-02 1.5575701e-02 1.4996317e-02
 1.4326520e-02 1.3462266e-02 1.1159586e-02 1.0136808e-02 9.8501900e-03
 8.6968569e-03 8.6978786e-03 7.8356685e-03 5.7884832e-03 5.0525339e-03
 5.5325846e-03 4.3480191e-03 4.2531961e-03 4.3486333e-03 3.7097572e-03
 3.0697561e-03 3.1024232e-03 3.0377037e-03 2.2066243e-03 2.0140086e-03
 2.0789320e-03 1.2471337e-03 1.6950325e-03 1.2470306e-03 1.5349799e-03
 1.0771881e-03 1.0875543e-03 7.5611024e-04 1.2380523e-03 6.6762709e-04
 7.9089811e-04 1.1154487e-03 6.3933898e-04 5.7800626e-04 6.2426768e-04
 4.3993708e-04 7.6890923e-04 1.3143705e-04 6.2151265e-04 5.2529899e-04
 4.3355225e-04 3.9954385e-04 7.1016466e-04 5.3354801e-04 3.5544269e-04
 1.5238134e-04 4.7974521e-04 1.7777823e-04 2.7414013e-04 9.6814794e-05
 7.001

In [ ]:
import numpy as np

arr = np.array([
1,8,35,5,7,20,2,14,2,3,36,2,2,12,23,23,4,6,5,1,6,6,1,16,
8,10,3,14,10,3,11,4,2,26,13,1,5,26,9,2,45,10,1,8,9,2,1,4,
7,35,10,1,26,3,2,7,5,14,5,20,5,1,16,3,6,13,4,20,27,2,2,37,
13,2,8,16,11,12,5,14,22,1,1,2,8,2,12,6,11,3,10,16,23,6,30,11,
1,7,11,32,23,4,1,3,6,41,22,15,3,17,4,12,11,4,16,1,5,7,12,5,
5,13,8,10,10,46,5,10,28,28,9,13,16,4,32,24,1,27,4,11,23,2,7,5,
5,6,3,2,11,3,1,12,4,29,19,17,4,23,1,6,9,15,18,4,18,5,20,7,
13,7,7,3,5,3,4,2,14,32,4,9,3,8,3,3,1,6,13,6,20,15,1,7,
2,7,11,10,8,1,12,63,1,11,17,3,21,12,37,3,19,1,19,1,18,4,13,5,
7,7,14,5,3,18,9,4,33,17,5,45,15,6,3,4,10,15,7,7,6,17,28,1,
6,7,5,1,1,30,14,3,12,48,13,2,7,15,3,13,5,1,1,8,5,1,1,26,
15,16,35,9,24,2,17,1,1,3,10,19,1,12,21,22,12,2,20,6,6,6,1,18,
4,25,7,6,1,8,19,2,1,7,3,2,5,0,0,0,0,0,0,0,0,0,0,0
])

n_zero = np.sum(arr == 0)
n_one = np.sum(arr == 1)

print("Number of zeros:", n_zero)
print("Number of ones :", n_one)

unique, counts = np.unique(arr, return_counts=True)
# print("\nFull distribution:")
# print(dict(zip(unique, counts)))

print("\nArray info:")
print("size :", arr.size)
print("min  :", arr.min())
print("max  :", arr.max())

Number of zeros: 11
Number of ones : 35

Array info:
size : 312
min  : 0
max  : 63


# .